# VisoSwap — Free GPU Runner on Kaggle

Run VisoSwap Studio on Kaggle's free GPU (NVIDIA T4 or P100).

### Instructions:
1. In the right panel under **Settings**:
   - **Accelerator**: Select **GPU T4 x2** (or **GPU P100**)
   - **Internet**: Toggle to **Internet on** (required for downloads and Cloudflare tunnel)
2. Click **Run All** (or run cells one-by-one).
3. Wait for the public Cloudflare tunnel URL printed at the bottom cell (e.g. `https://xxxx.trycloudflare.com`).
4. Open the link on any browser or mobile device (Safari/Chrome) to use VisoSwap with live stream swap and theater mode!

In [ ]:
# Step 1: Verify GPU Environment
!nvidia-smi

In [ ]:
# Step 2: Install System Dependencies & Cloudflare Tunnel
!apt-get update -qq && apt-get install -y -qq ffmpeg curl
!curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
# Step 3: Clone VisoSwap Repository (including pre-built Web UI)
import os, sys
if not os.path.exists('/kaggle/working/visoswap'):
    !git clone --depth 1 https://github.com/kaiser62/visoswap.git /kaggle/working/visoswap
else:
    %cd /kaggle/working/visoswap
    !git pull origin master

%cd /kaggle/working/visoswap
if '/kaggle/working/visoswap' not in sys.path:
    sys.path.insert(0, '/kaggle/working/visoswap')
!ls -la frontend/dist

In [ ]:
# Step 4: Install Python & Engine Dependencies with Clean GPU ONNX Runtime & CUDA libs
!pip uninstall -y onnxruntime onnxruntime-gpu
# onnxruntime-gpu >= 1.27 on PyPI is built for CUDA 13; Kaggle ships CUDA 12, so pin the last CUDA 12 line
!pip install --no-cache-dir "onnxruntime-gpu>=1.20,<1.27" nvidia-cudnn-cu12 nvidia-cuda-runtime-cu12

!pip install --no-cache-dir \
    fastapi uvicorn[standard] pydantic pydantic-settings aiosqlite httpx \
    python-multipart yt-dlp opencv-python Pillow ftfy regex numexpr onnxsim requests tqdm

# Ensure LD_LIBRARY_PATH and CUDA dynamic libraries are loaded into python
import os, sys, site, ctypes
from pathlib import Path

lib_paths = []
for p in site.getsitepackages():
    for sub in Path(p).glob('nvidia/*/lib'):
        if sub.is_dir():
            lib_paths.append(str(sub))
    torch_lib = Path(p) / 'torch' / 'lib'
    if torch_lib.is_dir():
        lib_paths.append(str(torch_lib))
lib_paths.extend(['/usr/local/cuda/lib64', '/usr/local/cuda/targets/x86_64-linux/lib'])

cur_ld = os.environ.get('LD_LIBRARY_PATH', '')
os.environ['LD_LIBRARY_PATH'] = ':'.join(lib_paths) + (f':{cur_ld}' if cur_ld else '')

rtld_mode = getattr(ctypes, 'RTLD_GLOBAL', 0)
import torch
for d in lib_paths:
    for name in ['libcudart.so.12', 'libcublasLt.so.12', 'libcublas.so.12', 'libcudnn_ops.so.9', 'libcudnn_cnn.so.9', 'libcudnn.so.9']:
        cand = os.path.join(d, name)
        if os.path.exists(cand):
            try:
                ctypes.CDLL(cand, mode=rtld_mode)
            except Exception:
                pass

import onnxruntime as ort
print('Available providers:', ort.get_available_providers())
assert 'CUDAExecutionProvider' in ort.get_available_providers(), 'CUDAExecutionProvider not detected!'

In [ ]:
# Step 5: Choose Model Download Option
# - 'default': Download only 7 models needed for default face swapping (~1.3 GB, FAST ~30 sec)
# - 'all': Download all 56 models (~12 GB, complete offline catalog)
MODEL_SUBSET = 'default'  # Change to 'all' if you want every optional model/enhancer

import os, sys
from pathlib import Path

if '/kaggle/working/visoswap' not in sys.path:
    sys.path.insert(0, '/kaggle/working/visoswap')
%cd /kaggle/working/visoswap

models_dir = Path('/kaggle/working/visoswap/model_assets_owned')
models_dir.mkdir(parents=True, exist_ok=True)
os.environ['MODELS_DIR'] = str(models_dir)
os.environ['MODELS_SUBSET'] = MODEL_SUBSET

from visoswap.models.bootstrap import repair, verify, FAST
from visoswap.models import manifest

print(f'Starting model bootstrap (subset={MODEL_SUBSET})...')
res = repair(models_dir=models_dir, mode=FAST, subset=MODEL_SUBSET)
print(f'Bootstrap status: {"OK" if res.ok else "INCOMPLETE"}')
print(f'Required models checked: {res.required_checked}, Present: {len(res.present)}')
for entry in res.present:
    print(f'  ✓ {entry.name} ({entry.path.name})')
if not res.ok:
    print('Missing entries:', [e.name for e in res.absent])
    raise RuntimeError('Model download incomplete!')

# Test CUDAExecutionProvider and IOBinding directly on RetinaFace
import torch, numpy as np, onnxruntime as ort
test_model = models_dir / 'det_10g.onnx'  # RetinaFace (see visoswap/models/models_data.py)
sess = ort.InferenceSession(str(test_model), providers=['CUDAExecutionProvider'])
print('Active session providers:', sess.get_providers())
assert 'CUDAExecutionProvider' in sess.get_providers(), f'CUDAExecutionProvider failed: {sess.get_providers()}'

io = sess.io_binding()
dummy = torch.zeros((1, 3, 640, 640), dtype=torch.float32, device='cuda')
io.bind_input(sess.get_inputs()[0].name, device_type='cuda', device_id=0, element_type=np.float32, shape=tuple(dummy.size()), buffer_ptr=dummy.data_ptr())
for out in sess.get_outputs():
    io.bind_output(out.name, 'cuda')
torch.cuda.synchronize()
sess.run_with_iobinding(io)
print('✅ CUDAExecutionProvider & GPU IOBinding verified working perfectly!')

In [ ]:
# Step 6: Launch Cloudflare Tunnel and Start VisoSwap Backend Server
import subprocess
import time
import re
import os
import sys

if '/kaggle/working/visoswap' not in sys.path:
    sys.path.insert(0, '/kaggle/working/visoswap')
%cd /kaggle/working/visoswap

# Ensure nvidia-cudnn-cu12 is present before starting
try:
    import nvidia.cudnn
except ImportError:
    print('⚡ Installing nvidia-cudnn-cu12 for GPU acceleration...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', 'nvidia-cudnn-cu12', 'nvidia-cuda-runtime-cu12'])

os.environ['MODELS_DIR'] = '/kaggle/working/visoswap/model_assets_owned'
os.environ['MODELS_VERIFY_MODE'] = 'fast'
# Each generation worker owns a full set of ORT sessions on GPU 0; the default of 4
# runs a 16 GB T4 out of memory, so keep it to 2 on Kaggle.
os.environ['GENERATION_CONCURRENCY'] = '2'
os.environ['MODELS_SUBSET'] = globals().get('MODEL_SUBSET', 'default')

# 1. Start Cloudflare Tunnel in background
tunnel_log = open('/kaggle/working/tunnel.log', 'w')
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT
)

# 2. Extract public trycloudflare.com URL
print('Waiting for Cloudflare Tunnel to connect...')
public_url = None
for _ in range(30):
    time.sleep(1)
    if os.path.exists('/kaggle/working/tunnel.log'):
        content = open('/kaggle/working/tunnel.log').read()
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
        if match:
            public_url = match.group(0)
            break

if public_url:
    print('=' * 65)
    print('🚀 VisoSwap is LIVE! Access your Web UI at:')
    print(f'👉 {public_url}')
    print(f'👉 Mobile UI: {public_url}/mobile')
    print('=' * 65)
else:
    print('Could not find tunnel URL yet. Check /kaggle/working/tunnel.log')

# 3. Run FastAPI/Uvicorn server with LD_LIBRARY_PATH preserved
!export LD_LIBRARY_PATH=$(python -c "import site, os, glob; paths = glob.glob(site.getsitepackages()[0] + '/nvidia/*/lib') + [site.getsitepackages()[0] + '/torch/lib', '/usr/local/cuda/lib64']; print(':'.join(p for p in paths if os.path.exists(p)))"):$LD_LIBRARY_PATH && python -m uvicorn backend.main:app --host 0.0.0.0 --port 8000
